In [1]:
EPOCHS = 15000
LR = 1E-4
BATCH_SIZE = 1

In [2]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")

Using MPS (Apple Silicon)
Device: mps


In [3]:
from radiance_field_project.data.datamodule import UCO3DDataModule

dm = UCO3DDataModule(
    batch_size=1,
    num_workers=0,
    image_height=512,
    image_width=512
)

torch.manual_seed(42)
train_loader = dm.train_loader()

print(f"Dataset size: {len(train_loader.dataset)}")
print(f"Num batches:  {len(train_loader)}")

Dataset size: 9399
Num batches:  9399


In [4]:
from radiance_field_project.models.simple_autoencoder import SimpleAutoencoder

network = SimpleAutoencoder().to(device)


In [5]:
import torch
from torch.optim import AdamW
import torch.nn.functional as F

from radiance_field_project.data.gaussian_features import batch_to_features

optimizer = AdamW(network.parameters(), lr=LR)

sample = next(iter(train_loader))

features, batchidx = batch_to_features(sample)
features = features.to(device)

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    recon = network(features)
    loss = F.mse_loss(features, recon)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Current loss = {loss.item():.6f}")


/Users/leonardkuch/Documents/Universitaet/Master/IS/ADL4CV/radiance-field-project/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch 0: Current loss = 10707.233398
Epoch 100: Current loss = 10363.905273
Epoch 200: Current loss = 9966.768555
Epoch 300: Current loss = 9455.256836
Epoch 400: Current loss = 8773.816406
Epoch 500: Current loss = 8033.961426
Epoch 600: Current loss = 7237.042480
Epoch 700: Current loss = 6405.689941
Epoch 800: Current loss = 5582.348633
Epoch 900: Current loss = 4814.501953
Epoch 1000: Current loss = 4139.678223
Epoch 1100: Current loss = 3580.250977
Epoch 1200: Current loss = 3131.955078
Epoch 1300: Current loss = 2771.607666
Epoch 1400: Current loss = 2481.333740
Epoch 1500: Current loss = 2248.455078
Epoch 1600: Current loss = 2062.937012
Epoch 1700: Current loss = 1914.284668
Epoch 1800: Current loss = 1793.737183
Epoch 1900: Current loss = 1691.871826
Epoch 2000: Current loss = 1602.313110
Epoch 2100: Current loss = 1519.964355
Epoch 2200: Current loss = 1441.234985
Epoch 2300: Current loss = 1365.800415
Epoch 2400: Current loss = 1292.094971
Epoch 2500: Current loss = 1220.207

KeyboardInterrupt: 

In [ ]:
from uco3d import render_splats
from radiance_field_project.data.gaussian_features import features_to_gaussians

from radiance_field_project.utils.splat_visualizer import SplatRenderVisualizer


render_colors_gt, render_alphas_gt, render_info_gt = render_splats(
    cameras=sample["camera"],
    splats=sample["sequence_gaussian_splats"],
    render_size=[512, 512]
)

SplatRenderVisualizer.show(render_colors=render_colors_gt, render_alphas=render_alphas_gt)

In [ ]:
network.eval()

with torch.no_grad():
    recon = network(features)

sequence_gaussians_splats_reconstructed = features_to_gaussians(recon)

render_colors_recon, render_alphas_recon, render_info_recon = render_splats(
    cameras=sample["camera"],
    splats=sequence_gaussians_splats_reconstructed,
    render_size=[512, 512]
)

SplatRenderVisualizer.show(render_colors=render_colors_recon, render_alphas=render_alphas_recon)